# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

from iacs.audit_system import (
    AuditRunner,
    RequirementCoverageAudit,
    TraceabilityAudit,
    TodoAudit,
)
from iacs.io_system import IOSystem
from iacs.registry import Registry

## Load components.yaml

In [ ]:
io = IOSystem()
components_df = io.read_entity_centered_file(Path("../components/components.yaml"))
registry = Registry.from_entity_centered(components_df)

print(f"Loaded {len(components_df)} components from {components_df['entity_id'].nunique()} entities")
print(f"Component types: {registry.component_types}")

## Run All Audits

In [ ]:
runner = AuditRunner(audits=[
    RequirementCoverageAudit(),
    TraceabilityAudit(),
    TodoAudit(),
])

results = runner.run(registry)

print(f"All audits passed: {runner.all_passed}")
print()
for name, result in results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{name}: {status}")

## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [ ]:
req_result = results["requirement_coverage"]
print(f"Passed: {req_result.passed}")
print(f"Uncovered requirements: {len(req_result.entities)}")
print()
if req_result.entities:
    print("Entities missing implementations:")
    for entity in sorted(req_result.entities):
        print(f"  - {entity}")

## Traceability Audit

Checks that all entities trace back to requirements.

In [ ]:
trace_result = results["traceability"]
print(f"Passed: {trace_result.passed}")
print(f"Orphan entities: {len(trace_result.entities)}")
print()
if trace_result.entities:
    print("Entities not tracing to requirements:")
    for entity in sorted(trace_result.entities):
        print(f"  - {entity}")

## Todo Audit

Reports outstanding todos.

In [ ]:
todo_result = results["todo"]
print(f"Passed: {todo_result.passed}")
print(f"Entities with todos: {len(todo_result.entities)}")
print()
if todo_result.messages:
    print("Outstanding todos:")
    for msg in todo_result.messages:
        print(f"  - {msg}")